# Introduction

This notebook uses Kahneman-Tversky Optimization (KTO) to fine-tune OpenAI Whisper model.

Input dataset is created by `1-create-datasets.ipynb`, which extracts Whisper features before uploading to Hugging Face.

The dataset splits must include:
- `input_features`
- `labels`
- `label` (1 = desirable, 0 = undesirable)

NOTE: HF TRL's KTOTrainer was evaluated for use, but it is primarily designed for causal language models (e.g. LLM models).
This notebook implements a custom KTO objective locally.

# Set parameters, hyper-parameters

## KTO

For `parameter_trainer_kto_beta` a useful sweep would be:
`[0.1, 0.3, 0.5, 1.0]`

Interpretation:
- `0.1`: conservative, possibly too weak
- `0.3-0.5`: reasonable starting range for sequence-level rewards (`beta * sum of content-token log-ratios`)
- `1.0`: stronger signal, but higher risk of instability or overfitting

`reward = beta * (policy_logps - reference_logps)`

should typically reach roughly `0.5-2.0` for correctly classified examples.
If rewards remain near zero, increase beta; if the loss saturates or WER worsens, reduce it.

- 2026.10.10: Previous training with beta=0.3 produced near-zero rewards and negative reward margins, indicating the KTO signal was too weak. Beta increased to 0.5 for next runs.
- 2026.10.26: Beta set to 1. If later evaluations keep showing a rising margin while WER worsens, consider a lower beta, such as 0.5, for the next run. 

Checkpoints are selected on held-out WER (desirable transcripts only), not KTO `eval_loss`.
Also watch TensorBoard for `kto_rewards_desirable` vs `kto_rewards_undesirable`.

In [ ]:
# parameters for KTO

# @todo: assess if the following parameters shall be left here as parametric
# or directly fixed in the cells below, as they are not expected to be changed by the user
# - parameter_trainer_learning_rate
# REDUCED from 5e-6 to 1e-6 for more stable training
parameter_trainer_learning_rate = 1e-6
# parameter_trainer_learning_rate = 5e-6

# INCREASED from 0.3 to 0.5 to strengthen the KTO signal
# Previous value of 0.3 produced near-zero rewards and negative reward margins
# parameter_trainer_kto_beta                  = 0.5
parameter_trainer_kto_beta                = 1

# None = set after dataset load so each class contributes equally to the loss
parameter_trainer_desirable_weight          = None
parameter_trainer_undesirable_weight        = None

# EMA of KL(policy || reference) used as KTO z0; avoids collapsing on single-class batches
parameter_trainer_kto_kl_ema_decay          = 0.9
parameter_trainer_kto_mask_special_tokens   = True

## Dataset

In [ ]:
# Hugging Face datasets
# these must be available on the Hugging Face Hub

# These datasets will be used for training the model
# If more than one dataset is specified, the model will be trained on the union of all datasets
# Thes datasets must contain the following splits: 'train', 'test', 'eval'
HF_DATASETS_TRAINING = [
    "luca-r/chiara_dataset-kto_20261112"
]

## Model

In [ ]:
# - OpenAi Whisper starting checkpoint
# -- Local
parameter_basemodel_path = "/home/luca/Development/chiara-speech2text_models/chiara_whisper_20261026"
# -- HuggingFace Hub
#parameter_basemodel_path = "openai/whisper-tiny"
#parameter_basemodel_path = "openai/whisper-base"
#parameter_basemodel_path = "openai/whisper-small"
#parameter_basemodel_path = "openai/whisper-medium"
#parameter_basemodel_path = "openai/whisper-large"
#parameter_basemodel_path = "openai/whisper-large-v3"
#parameter_basemodel_path = "openai/whisper-large-v3-turbo"
#parameter_basemodel_path = f"{HF_USER}/chiara_whisper-large-v3-turbo_20260624"


# - N bins for features extractor
#  parameter_model_n_mel_bins=80 is the standard for openai/whisper-small. This is compatibile with FasterWhisper inference
#  parameter_model_n_mel_bins=128 is the standard for openai/whisper-large-v3-turbo. ?This is NOT compatible with FasterWhisper inference?
parameter_model_n_mel_bins = 128


# - ASR model language
# from transformers.models.whisper import tokenization_whisper
# languages_list = list(tokenization_whisper.TO_LANGUAGE_CODE.values()) # list of supported languages
parameter_model_language  = 'it'


# 
parameter_model_task = 'transcribe' # 'transcribe' or 'translate'


# - Print
print(f"Base model: {parameter_basemodel_path}")
print(f"Language:   {parameter_model_language}")

## Trainer

In [ ]:
# - parameter_trainer_batch_size
# this directly affects footprint on the GPU RAM
# Depending on available RAM, it is possible that you will encounter a CUDA "out-of-memory" error when you start training.
# if short on GPU RAM, reduce this value incrementally by factors of 2
# (and potentially employ gradient_accumulation_steps to compensate)
#parameter_trainer_batch_size = 1    # for whisper-large-v3, on Google Colab-hosted T4 GPU
parameter_trainer_batch_size = 12
#parameter_trainer_batch_size = 16   # batch_size=16 equals to  ~90 GB of GPU RAM (for whisper-large-v3, on nvidia dgx spark)
#parameter_trainer_batch_size = 18  # batch_size=18 equals to  ~99 GB of GPU RAM (for whisper-large-v3, on nvidia dgx spark)
#parameter_trainer_batch_size = 20  # batch_size=20 system crashes "out of memory"
# If CUDA OOM after unfreezing the decoder, lower this (decoder Adam states add memory).


# - parameter_trainer_n_epochs default is 3
# This parameter has a big influence on training time
# Plots for SFT training show that after ~3 epochs, the models converge
parameter_trainer_n_epochs = 2
#parameter_trainer_n_epochs = 0.5 # <--------------- for quick tests


# - parameter_trainer_warmup_steps
parameter_trainer_warmup_steps  = 10


# - eval_batch_size
parameter_trainer_eval_batch_size = 8


# - max_gen_len
# set this as short as possible for your data
# if not set, it defaults to the max_length value of the model configuration
parameter_trainer_max_gen_len   = 48


# - layers to be updated
# KTO scores decoder sequence log-probs of good vs bad transcripts, so the decoder
# (or at least cross-attention) must be trainable. Encoder-only KTO keeps rewards ~0.
parameter_trainer_update_encoder    = True
parameter_trainer_update_decoder    = True
parameter_trainer_update_projection = True


# - early stopping## Credentials
parameter_early_stopping_patience   = 20    # stop after N evaluations without improvement
#parameter_early_stopping_patience   = 1     # <--------------- for quick tests

parameter_early_stopping_threshold  = 0.025  # minimum improvement required

## Directories

In [ ]:
# Directory for output model
DIR_MODELS_OUTPUT           = f"/home/luca/Development/chiara-speech2text_models/temp/"

## Credentials

In [ ]:
# Credentials

# env file with configs and credentials
ENV_FILE = "/home/luca/Development/chiara-speech2text_envs/.env"

# ToDos

- ...

# Init notebook

In [ ]:
# Install packages

%pip install --quiet pip
%pip install --quiet datasets[audio] accelerate evaluate jiwer tensorboard
%pip install --quiet datasets
%pip install --quiet tokenizers
%pip install --quiet librosa
%pip install --quiet transformers
%pip install --quiet ipywidgets
%pip install --quiet huggingface_hub
%pip install --quiet cupy-cuda13x
#%pip install --quiet Levenshtein # https://huggingface.co/spaces/Natooz/levenshtein
%pip install --quiet tensorboard

# run "nvidia-smi" and check CUDA version
# setup torch with the matching CUDA version, e.g. cu130 for CUDA 13.0
%pip install --quiet torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu130

In [ ]:
# Import variables from .env file

# Load the variables from the .env file into the environment
from dotenv import load_dotenv
load_dotenv(ENV_FILE)

In [ ]:
# Load Hugging Face variables from environment
import os
HF_USER     = os.environ.get("HF_USER")
HF_TOKEN    = os.environ.get("HF_TOKEN")

#from google.colab import userdata
#HF_USER="luca-r"
#HF_TOKEN = userdata.get('HF_TOKEN')

print(f"I will use the HF user: {HF_USER}, token: {HF_TOKEN}")

In [ ]:
import sys
import torch

print(sys.executable)
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

assert torch.cuda.is_available(), "CUDA GPU is unavailable"

In [ ]:
# Imports

import os
import soundfile as sf
import numpy as np
from datetime import datetime
from collections import defaultdict
from dataclasses import dataclass
import json
import numbers
from datasets import DatasetDict, Dataset

import ipywidgets as widgets
from IPython.display import Audio
from IPython.display import display

import evaluate
from transformers import Seq2SeqTrainingArguments
from transformers import Seq2SeqTrainer
from transformers import WhisperForConditionalGeneration
from transformers import WhisperProcessor
from transformers import (
    EarlyStoppingCallback
)
from transformers.models.whisper import tokenization_whisper

import torch
from accelerate import Accelerator

from huggingface_hub import login, HfApi
from huggingface_hub.utils import HfHubHTTPError # Import HfHubHTTPError directly from utils

import soundfile as sf
import torch
from datasets import DatasetDict, load_dataset
from transformers import (
    EarlyStoppingCallback,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    WhisperForConditionalGeneration,
    WhisperProcessor,
)

In [ ]:
# get today's date in format YYYYMMDD
# this will be used to filename the ouptut from this notebook
from datetime import datetime
string_date = datetime.now().strftime("%Y%m%d")

In [ ]:
import os

# create directory for final model
#DIR_WHISPER_MODEL_OUTPUT = f"{DIR_MODELS_OUTPUT}/chiara_whisper-large-v3-turbo_kto_{string_date}"
#DIR_WHISPER_MODEL_OUTPUT = f"{DIR_MODELS_OUTPUT}/chiara_{base_model_name}_{string_date}"
DIR_WHISPER_MODEL_OUTPUT = f"{DIR_MODELS_OUTPUT}/chiara_whisper_{string_date}"

if os.path.exists(DIR_WHISPER_MODEL_OUTPUT):
    i = 1
    while os.path.exists(DIR_WHISPER_MODEL_OUTPUT + f"_{i}"):
        i += 1
    DIR_WHISPER_MODEL_OUTPUT = DIR_WHISPER_MODEL_OUTPUT + f"_{i}"
os.makedirs(DIR_WHISPER_MODEL_OUTPUT, exist_ok=True)

# One directory is reused for the latest intermediary model.
DIR_WHISPER_INTERMEDIARY_MODEL = os.path.join(
    DIR_MODELS_OUTPUT, "chiara_whisper-large-v3-turbo_kto_intermediary"
)

# Temp folders for Whisper
DIR_WHISPER_LOG     = os.path.join(DIR_MODELS_OUTPUT, 'whisper-logs')
DIR_WHISPER_OUTPUT  = os.path.join(DIR_MODELS_OUTPUT, 'whisper-output')

# create temp folders for Whisper
!rm -rf {DIR_WHISPER_LOG}
!mkdir  {DIR_WHISPER_LOG}

In [ ]:
# Disable ipywidgets bars (they cause visualization errors with transforms library in notebooks)

import os
os.environ["HF_DATASETS_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"]      = "1"

import datasets
datasets.disable_progress_bars()

In [ ]:
# Login to Hugging Face (the processed dataset will be stored there)

from huggingface_hub import login, HfApi
from huggingface_hub.utils import HfHubHTTPError

# login to HF
login(token=HF_TOKEN) # Pass the token explicitly for consistency and clarity

# Verify login by attempting to retrieve user information
try:
    HfApi().whoami()
    print("Successfully logged in to Hugging Face Hub.")
except HfHubHTTPError as e:
    raise RuntimeError(f"Hugging Face login failed: {e}. Please ensure your 'hf_token' secret is valid and you have restarted the Colab runtime after setting it.")
except Exception as e:
    raise RuntimeError(f"An unexpected error occurred during Hugging Face login verification: {e}")

# cleanup
del HF_TOKEN

In [ ]:
# Empty GPU RAM

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("GPU RAM emptied.")
else:
    print("No GPU available, no GPU RAM to empty.")

In [ ]:
# Disable ipywidgets bars (they cause visualization errors with transforms library in notebooks)

import os
os.environ["HF_DATASETS_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"]      = "1"

import datasets
datasets.disable_progress_bars()

# Dataset

## Download from Hugging Face

In [ ]:
# download training dataset(s)
# datasets with ~6500 samples take ~6.5m to download

from datasets import DatasetDict, concatenate_datasets, load_dataset

if not HF_DATASETS_TRAINING: raise ValueError("HF_DATASETS_TRAINING must contain at least one dataset.")
datasets = [load_dataset(dataset_id) for dataset_id in HF_DATASETS_TRAINING]

# print info
for i, dataset in enumerate(datasets):
    print(f"Dataset #{i}: {HF_DATASETS_TRAINING[i]}")
    #print(f"  Entries: {len(dataset):,}")
    print(f"  Dataset: {dataset}")

## Concatenate different datasets 

In [ ]:
# check if all datasets have the same splits
splits = set(datasets[0].keys())
if any(set(dataset.keys()) != splits for dataset in datasets[1:]):
    raise ValueError("All datasets in HF_DATASETS_TRAINING must have the same splits.")

# Merge datasets into a single dataset
hf_dataset = DatasetDict({
    split: concatenate_datasets([dataset[split] for dataset in datasets])
    for split in datasets[0]
})
# hf_dataset.save_to_disk(DIR_DATA_HF)

## Compute class weights

In [ ]:
# Compute class weights for KTO training
# based on data labels

import time

def _kto_binary_label(example):
    if example.get("kto_label") is not None:
        return int(example["kto_label"])
    if example.get("label") is not None:
        return int(example["label"])
    raise KeyError("KTO dataset examples need 'kto_label' or 'label'")


train_split = hf_dataset["train"]
n_examples  = len(train_split)
print(f"KTO train split: {n_examples} examples, columns={list(train_split.column_names)}")
print(f"KTO train features: {train_split.features}")
print("Scanning train labels...")

train_kto_labels = []
progress_every   = max(n_examples // 20, 1)
t0 = time.perf_counter()
for i, example in enumerate(train_split):
    if i == 0:
        print(f"First example keys: {list(example.keys())}")
        print(f"First example kto_label={example.get('kto_label')!r}, label={example.get('label')!r}")
    train_kto_labels.append(_kto_binary_label(example))
    if (i + 1) % progress_every == 0 or (i + 1) == n_examples:
        elapsed = time.perf_counter() - t0
        rate    = (i + 1) / elapsed if elapsed > 0 else 0.0
        print(
            f"  scanned {i + 1}/{n_examples} "
            f"({100.0 * (i + 1) / n_examples:.1f}%) "
            f"in {elapsed:.1f}s ({rate:.1f} examples/s)",
            flush=True,
        )
print(f"Finished scanning labels in {time.perf_counter() - t0:.1f}s")

n_desirable         = sum(train_kto_labels)
n_undesirable       = len(train_kto_labels) - n_desirable
n_train             = max(n_desirable + n_undesirable, 1)

if parameter_trainer_desirable_weight   is None:
    parameter_trainer_desirable_weight      = n_train / (2.0 * max(n_desirable, 1))
if parameter_trainer_undesirable_weight is None:
    parameter_trainer_undesirable_weight    = n_train / (2.0 * max(n_undesirable, 1))

print(
    f"KTO train labels: desirable={n_desirable}, undesirable={n_undesirable}"
)
print(
    f"KTO class weights: desirable={parameter_trainer_desirable_weight:.4f}, "
    f"undesirable={parameter_trainer_undesirable_weight:.4f}"
)

# Train

## Configure training

In [ ]:
# Create processor

# --------------------------------------------------------------------------------
#                                  IMPORTANT
# --------------------------------------------------------------------------------
#
# make sure these parameters are the !same! as those in dataset-creation script
#
# --------------------------------------------------------------------------------


# Download Whisper checkpoint
print('Using Language: ', parameter_model_language)
parameter_model_task = "transcribe"
hf_processor = WhisperProcessor.from_pretrained(
    parameter_basemodel_path,
    language        =parameter_model_language,
    task            =parameter_model_task,
    num_mel_bins    =parameter_model_n_mel_bins)

In [ ]:
# Create model
# Download if not already present

hf_base_model = WhisperForConditionalGeneration.from_pretrained(parameter_basemodel_path)

In [ ]:
# Set model properties

hf_base_model = hf_base_model.to(torch.float32)

hf_base_model.generation_config.language           = parameter_model_language
hf_base_model.generation_config.task               = parameter_model_task
hf_base_model.generation_config.forced_decoder_ids = None
hf_base_model.config.forced_decoder_ids            = None

# Fix for AttributeError: 'GenerationConfig' object has no attribute 'lang_to_id'
if not hasattr(hf_base_model.generation_config, "lang_to_id"):
    if hasattr(hf_processor.tokenizer, "lang_code_to_id"):
        hf_base_model.generation_config.lang_to_id = hf_processor.tokenizer.lang_code_to_id
    else:
        # Fallback for Fast tokenizer if attribute is missing
        hf_base_model.generation_config.lang_to_id = {
            code: hf_processor.tokenizer.convert_tokens_to_ids(f"<|{code}|>")
            for code in tokenization_whisper.TO_LANGUAGE_CODE.values()
        }

# Fix for 'task_to_id'
if not hasattr(hf_base_model.generation_config, "task_to_id"):
    hf_base_model.generation_config.task_to_id = {
        task: hf_processor.tokenizer.convert_tokens_to_ids(f"<|{task}|>")
        for task in ["transcribe", "translate"]
    }

# to use gradient checkpointing
# hf_base_model.config.use_cache = False # Commented out as gradient checkpointing is causing issues

# Explicitly disable gradient checkpointing in the model config
hf_base_model.config.gradient_checkpointing_enabled = False

print('language set to:', hf_base_model.generation_config.language)

In [ ]:
# Define early-stopping callback
hf_trainer_early_stopping_callback = EarlyStoppingCallback(
    early_stopping_patience     = parameter_early_stopping_patience,
    early_stopping_threshold    = parameter_early_stopping_threshold,
)

In [ ]:
hf_processor = WhisperProcessor.from_pretrained(
        parameter_basemodel_path,
        language    =parameter_model_language,
        task        =parameter_model_task,
        num_mel_bins=parameter_model_n_mel_bins,
    )

In [ ]:
# Training arguments are reused for each dataset iteration.
parameter_trainer_eval_steps = 5
parameter_trainer_save_steps = 3*parameter_trainer_eval_steps # "parameter_trainer_save_steps" must be a multiple of "parameter_trainer_eval_steps"

hf_trainer_args = Seq2SeqTrainingArguments(
    output_dir                  =str(DIR_WHISPER_OUTPUT),
    #logging_dir                 =DIR_WHISPER_LOG,
    
    per_device_train_batch_size =parameter_trainer_batch_size,
    gradient_accumulation_steps =1,
    gradient_checkpointing      =False,
    fp16                        =False,
    num_train_epochs            =parameter_trainer_n_epochs,
    #
    lr_scheduler_type           ="constant_with_warmup",
    learning_rate               =parameter_trainer_learning_rate,
    warmup_steps                =parameter_trainer_warmup_steps,
    #
    eval_strategy               ="steps",
    eval_steps                  =parameter_trainer_eval_steps,
    per_device_eval_batch_size  =parameter_trainer_eval_batch_size,
    predict_with_generate       =True, # needed for WER/CER during eval
    generation_max_length       =parameter_trainer_max_gen_len,
    metric_for_best_model       ="wer",
    greater_is_better           =False,
    #
    save_steps                  =parameter_trainer_save_steps,
    save_total_limit            =2,
    save_strategy               ="best",
    logging_steps               =1,
    load_best_model_at_end      =True,
    #
    remove_unused_columns       =False,
    push_to_hub                 =False,
    eval_on_start               =True,
    #
    do_train                    =True, # Trains the model
    do_eval                     =True, # Evaluates on the validation set during training
    do_predict                  =True, # Runs a final evaluation on the test set post-training
    # multi CPU workers
    dataloader_num_workers          =2,
    dataloader_pin_memory           =True,
    dataloader_persistent_workers   =False,
    #
    report_to                   =["tensorboard"],
)

In [ ]:
# Define parameters to be updated
# KTO needs decoder log-probs to move; keep the decoder trainable unless you are
# deliberately running an encoder-only ablation.
hf_base_model.model.encoder.requires_grad_(parameter_trainer_update_encoder)
hf_base_model.model.decoder.requires_grad_(parameter_trainer_update_decoder)
hf_base_model.proj_out.requires_grad_(parameter_trainer_update_projection)

# This function counts the number of trainable parameters in the model.
def count_trainable_parameters(model):
    model_parameters = filter(lambda p: p.requires_grad, model.parameters())
    params = sum([np.prod(p.size()) for p in model_parameters])
    return params

print('encoder params to update/total:',    count_trainable_parameters(hf_base_model.model.encoder), hf_base_model.model.encoder.num_parameters())
print('decoder params to update/total:',    count_trainable_parameters(hf_base_model.model.decoder), hf_base_model.model.decoder.num_parameters())
print('overall # trainable parameters:',    count_trainable_parameters(hf_base_model))
print('overall # model parameters:',        hf_base_model.model.num_parameters())

## KTO utils

In [ ]:
# define a data collator that pads the inputs and labels for Whisper, while keeping the KTO label per sample

from typing import Any, Dict, List, Union
from pathlib import Path
from collections import defaultdict
import copy

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    """Pad Whisper inputs and labels while keeping the KTO label per sample."""

    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features  = [{"input_features": feature["input_features"]} for feature in features]
        batch           = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        label_features  = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch    = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels          = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"]     = labels
        batch["kto_label"]  = torch.tensor(
            [_feature_kto_label(feature) for feature in features],
            dtype=torch.float32,
        )
        return batch


def _feature_kto_label(feature: Dict[str, Any]) -> int:
    if feature.get("label") is not None:
        return int(feature["label"])
    if feature.get("kto_label") is not None:
        return int(feature["kto_label"])
    raise KeyError("KTO example is missing 'label' / 'kto_label'")


def _whisper_special_token_ids(processor: WhisperProcessor) -> list[int]:
    tokenizer = processor.tokenizer
    ids = set(tokenizer.all_special_ids)
    language_code = tokenization_whisper.TO_LANGUAGE_CODE.get(
        parameter_model_language, parameter_model_language
    )
    extra_tokens = [
        tokenizer.bos_token,
        tokenizer.eos_token,
        tokenizer.pad_token,
        tokenizer.unk_token,
        "<|startoftranscript|>",
        "<|endoftext|>",
        "<|transcribe|>",
        "<|translate|>",
        "<|notimestamps|>",
        "<|nospeech|>",
        f"<|{parameter_model_language}|>",
        f"<|{language_code}|>",
    ]
    for token in extra_tokens:
        if not token:
            continue
        token_id = tokenizer.convert_tokens_to_ids(token)
        if isinstance(token_id, int):
            ids.add(token_id)
    unk_id = tokenizer.unk_token_id
    return sorted(token_id for token_id in ids if token_id is not None and token_id != unk_id)


wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")


class WhisperKTOTrainer(Seq2SeqTrainer):
    """Seq2SeqTrainer using KTO with a frozen reference model for Whisper."""

    def __init__(
        self,
        *args: Any,
        beta: float = 0.1,
        desirable_weight: float = 1.0,
        undesirable_weight: float = 1.0,
        reference_model: WhisperForConditionalGeneration = None,
        kl_ema_decay: float = 0.9,
        mask_special_tokens: bool = True,
        special_token_ids: list[int] | None = None,
        timestamp_begin: int | None = None,
        **kwargs: Any,
    ) -> None:
        super().__init__(*args, **kwargs)
        self.beta = beta
        self.desirable_weight = desirable_weight
        self.undesirable_weight = undesirable_weight
        self.reference_model = reference_model
        self.kl_ema_decay = kl_ema_decay
        self.mask_special_tokens = mask_special_tokens
        self.special_token_ids = special_token_ids or []
        self.timestamp_begin = timestamp_begin
        self.kl_ema: torch.Tensor | None = None
        self._stored_metrics: Dict[str, Dict[str, list[float]]] = {
            "train": defaultdict(list),
            "eval": defaultdict(list),
        }
        self._eval_kto_labels: list[torch.Tensor] = []
        if self.compute_metrics is None:
            self.compute_metrics = self.compute_asr_metrics

    def _content_mask(self, labels: torch.Tensor) -> torch.Tensor:
        mask = labels.ne(-100)
        if self.mask_special_tokens and self.special_token_ids:
            special = torch.tensor(self.special_token_ids, device=labels.device, dtype=labels.dtype)
            mask = mask & ~torch.isin(labels, special)
        if self.timestamp_begin is not None:
            mask = mask & labels.lt(self.timestamp_begin)
        fallback = labels.ne(-100)
        empty_rows = mask.sum(dim=1) == 0
        if empty_rows.any():
            mask = mask.clone()
            mask[empty_rows] = fallback[empty_rows]
        return mask

    @staticmethod
    def _sequence_log_probability(
        logits: torch.Tensor,
        labels: torch.Tensor,
        mask: torch.Tensor,
    ) -> torch.Tensor:
        token_log_probs = torch.log_softmax(logits, dim=-1)
        safe_labels     = labels.masked_fill(labels.eq(-100), 0)
        selected        = token_log_probs.gather(2, safe_labels.unsqueeze(-1)).squeeze(-1)
        return (selected * mask).sum(dim=1)

    def _store_metrics(self, split: str, metrics: Dict[str, torch.Tensor | float]) -> None:
        for key, value in metrics.items():
            if isinstance(value, torch.Tensor):
                if value.numel() == 0:
                    continue
                value = value.detach().float().mean().cpu().item()
            self._stored_metrics[split][key].append(float(value))

    def compute_loss(
        self,
        model: WhisperForConditionalGeneration,
        inputs: Dict[str, torch.Tensor],
        return_outputs: bool = False,
        num_items_in_batch: int | None = None,
    ) -> torch.Tensor | tuple[torch.Tensor, Dict[str, torch.Tensor]]:
        del num_items_in_batch
        labels          = inputs["labels"]
        kto_label       = inputs["kto_label"].to(model.device).float()
        model_inputs    = {key: value for key, value in inputs.items() if key != "kto_label"}

        policy_outputs  = model(**model_inputs, use_cache=False)
        content_mask    = self._content_mask(labels)
        policy_logps    = self._sequence_log_probability(policy_outputs.logits, labels, content_mask)

        with torch.no_grad():
            ref_outputs = self.reference_model(**model_inputs, use_cache=False)
            reference_logps = self._sequence_log_probability(ref_outputs.logits, labels, content_mask)

        log_ratio = policy_logps - reference_logps
        kl_batch  = log_ratio.detach().mean()
        if model.training:
            if self.kl_ema is None:
                self.kl_ema = kl_batch
            else:
                self.kl_ema = self.kl_ema_decay * self.kl_ema + (1.0 - self.kl_ema_decay) * kl_batch
        z0 = self.kl_ema if self.kl_ema is not None else kl_batch
        z0 = z0.clamp(min=0)

        # r_theta = log pi - log pi_ref; z0 tracks KL so the preference is relative
        desirable           = kto_label > 0.5
        desirable_loss      = -torch.nn.functional.logsigmoid(self.beta * (log_ratio - z0))
        undesirable_loss    = -torch.nn.functional.logsigmoid(self.beta * (z0 - log_ratio))
        weights = torch.where(
            desirable,
            torch.full_like(log_ratio, self.desirable_weight),
            torch.full_like(log_ratio, self.undesirable_weight),
        )
        losses  = torch.where(desirable, desirable_loss, undesirable_loss) * weights
        loss    = losses.mean()
        rewards = (self.beta * log_ratio).detach()

        split = "train" if model.training else "eval"
        self._store_metrics(split, {"kto_z0": z0.detach()})
        if desirable.any():
            self._store_metrics(split, {"kto_rewards_desirable": rewards[desirable]})
        if (~desirable).any():
            self._store_metrics(split, {"kto_rewards_undesirable": rewards[~desirable]})
        if desirable.any() and (~desirable).any():
            self._store_metrics(
                split,
                {"kto_reward_margin": rewards[desirable].mean() - rewards[~desirable].mean()},
            )

        if return_outputs:
            return loss, {"logits": policy_outputs.logits, "kto_rewards": rewards}
        return loss

    def prediction_step(
        self,
        model: WhisperForConditionalGeneration,
        inputs: Dict[str, torch.Tensor],
        prediction_loss_only: bool,
        ignore_keys: List[str] | None = None,
        **gen_kwargs: Any,
    ):
        del ignore_keys
        inputs = self._prepare_inputs(inputs)
        kto_label = inputs.get("kto_label")
        if kto_label is not None:
            self._eval_kto_labels.append(kto_label.detach().cpu())

        with torch.no_grad():
            loss = self.compute_loss(model, inputs)

        if prediction_loss_only or not self.args.predict_with_generate:
            return (loss.detach(), None, inputs.get("labels"))

        if len(gen_kwargs) == 0 and hasattr(self, "_gen_kwargs"):
            gen_kwargs = self._gen_kwargs.copy()
        gen_kwargs.pop("num_beams", None) if gen_kwargs.get("num_beams") is None else None
        gen_kwargs.pop("max_length", None) if gen_kwargs.get("max_length") is None else None

        generation_inputs = {
            key: value
            for key, value in inputs.items()
            if key in ("input_features", "attention_mask")
        }
        generated_tokens = self.model.generate(**generation_inputs, **gen_kwargs)

        labels = inputs.get("labels")
        gen_config = self.model.generation_config
        pad_to = gen_config.max_length
        if gen_config.max_new_tokens is not None:
            pad_to = max(pad_to or 0, gen_config.max_new_tokens + 1)
        if pad_to and generated_tokens.shape[-1] < pad_to:
            generated_tokens = self._pad_tensors_to_max_len(generated_tokens, pad_to)
        if labels is not None and pad_to and labels.shape[-1] < pad_to:
            labels = self._pad_tensors_to_max_len(labels, pad_to)
        return (loss.detach(), generated_tokens, labels)

    def evaluate(self, *args: Any, **kwargs: Any):
        self._eval_kto_labels = []
        return super().evaluate(*args, **kwargs)

    def log(self, logs: Dict[str, float], *args: Any, **kwargs: Any):
        split = "eval" if any(str(key).startswith("eval_") for key in logs) else "train"
        stored = self._stored_metrics[split]
        for key, values in stored.items():
            if not values:
                continue
            metric_name = key if split == "train" or key.startswith("eval_") else f"eval_{key}"
            logs[metric_name] = sum(values) / len(values)
        stored.clear()
        return super().log(logs, *args, **kwargs)

    def compute_asr_metrics(self, pred) -> Dict[str, float]:
        pred_ids = pred.predictions
        label_ids = pred.label_ids
        if isinstance(pred_ids, tuple):
            pred_ids = pred_ids[0]
        pred_ids = np.asarray(pred_ids)
        label_ids = np.asarray(label_ids)

        kto_labels = None
        if self._eval_kto_labels:
            kto_labels = torch.cat([tensor.reshape(-1) for tensor in self._eval_kto_labels]).numpy()
            self._eval_kto_labels = []
            if len(kto_labels) != len(label_ids):
                kto_labels = None

        pad_id = hf_processor.tokenizer.pad_token_id
        label_ids = np.where(label_ids == -100, pad_id, label_ids)
        pred_str = hf_processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
        label_str = hf_processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

        # Undesirable KTO rows use rejected transcripts, which are not ASR references.
        if kto_labels is not None:
            keep = kto_labels > 0.5
            if keep.any():
                pred_str = [text for text, flag in zip(pred_str, keep) if flag]
                label_str = [text for text, flag in zip(label_str, keep) if flag]

        if not pred_str:
            return {"wer": float("nan"), "cer": float("nan")}

        return {
            "wer": 100 * wer_metric.compute(predictions=pred_str, references=label_str),
            "cer": 100 * cer_metric.compute(predictions=pred_str, references=label_str),
        }


def _prepare_example(example: Dict[str, Any], processor: WhisperProcessor) -> Dict[str, Any]:
    audio, sampling_rate = sf.read(example["audio_data"], dtype="float32")
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    inputs = processor.feature_extractor(audio, sampling_rate=sampling_rate)
    labels = processor.tokenizer(example["kto_transcript"]).input_ids
    return {
        "input_features": inputs.input_features[0],
        "labels": labels,
        "label": _feature_kto_label(example),
    }

In [ ]:
# Create a frozen reference model for KTO
# The reference model is a snapshot of the policy before training,
# used to compute baseline log-probabilities for the KTO objective.
# NOTE: this doubles GPU memory usage. If you run out of memory,
#       reduce parameter_trainer_batch_size (e.g. to 8-10).
#
# Important: the policy and reference model must remain on the same device.
# If the base model is moved to GPU after the deep copy, the reference model
# can otherwise stay on CPU and trigger CUDA tensor mismatches during training.

import copy

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
hf_base_model = hf_base_model.to(device=device, dtype=torch.float32)
hf_reference_model = copy.deepcopy(hf_base_model).to(device=device, dtype=torch.float32)
hf_reference_model.requires_grad_(False)
hf_reference_model.eval()


In [ ]:
# Define trainer

# collator
hf_data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor               = hf_processor,
    decoder_start_token_id  = hf_base_model.config.decoder_start_token_id,
)

# trainer
hf_trainer = WhisperKTOTrainer(
    args                =hf_trainer_args,
    model               =hf_base_model,
    reference_model     =hf_reference_model,
    train_dataset       =hf_dataset["train"],
    eval_dataset        =hf_dataset["test"],
    data_collator       =hf_data_collator,
    processing_class    =hf_processor,
    beta                =parameter_trainer_kto_beta,
    desirable_weight    =parameter_trainer_desirable_weight,
    undesirable_weight  =parameter_trainer_undesirable_weight,
    kl_ema_decay        =parameter_trainer_kto_kl_ema_decay,
    mask_special_tokens =parameter_trainer_kto_mask_special_tokens,
    special_token_ids   =_whisper_special_token_ids(hf_processor),
    timestamp_begin     =getattr(hf_processor.tokenizer, "timestamp_begin", None),
    compute_metrics     =None,
    callbacks           =[hf_trainer_early_stopping_callback],
)

# cleanup
del Any, Dict, List, Union

## Prepare training

In [ ]:
# Set `PYTORCH_CUDA_ALLOC_CONF`
# Setting `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True` can help mitigate
# CUDA out-of-memory errors by enabling a more flexible memory allocation
# strategy in PyTorch. This allows PyTorch to reserve and manage GPU memory
# more efficiently, potentially preventing fragmentation issues that lead
# to OOM errors, especially when dealing with varying memory demands during training.

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print(f"PYTORCH_CUDA_ALLOC_CONF set to: {os.environ.get('PYTORCH_CUDA_ALLOC_CONF')}")

In [ ]:
# Start tensorboard
# Tensorboard shows metrics from trainer.train() in the next cell.
# Use the URL/port printed by this cell (often 6006). If it says "Reusing TensorBoard on port N",
# open http://localhost:N — a leftover process on 6006 may be watching a different/empty logdir.
# If ssh'd / in Cursor, forward that same port in the Ports tab.
# Logs are written under DIR_WHISPER_OUTPUT/runs/ (default Hugging Face logging_dir).

%load_ext tensorboard
%reload_ext tensorboard
%tensorboard --logdir {DIR_WHISPER_OUTPUT}

In [ ]:
# Empty GPU RAM
import torch

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("GPU RAM emptied.")
else:
    print("No GPU available, no GPU RAM to empty.")

In [ ]:
# Empty GPU RAM
import torch

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("GPU RAM emptied.")
else:
    print("No GPU available, no GPU RAM to empty.")

# Move models to the selected device (GPU)
device = torch.device("cuda:0")

## Evaluate model pre-training

In [ ]:
# eval dataset
# This dataset has NOT been seen during training
# the results are representative of model's performance on unseen / new data

'''
print('Evaluating best model on "eval" set, pre-training, language:', parameter_model_language)
results_eval_pre = hf_trainer.evaluate(hf_dataset["eval"], language=parameter_model_language)
print(results_eval_pre)
''' 

## Fine tune

In [ ]:
# Fine-tune model
# ...it takes time to train a model from scratch! e.g.:
# - ~2h30m for 30mins of input audio-data, 10 epochs
# - ~5h30m for 1h45mins of input audio-data, 10 epochs

# You can monitor resources with the following terminal commands
# watch -n 1 nvidia-smi
# btop

# @todo
# - fix warning: [transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


# define accelerator
hf_accelerator = Accelerator()

# start timer
start_time = datetime.now()

# Move models to the selected device (GPU)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
hf_base_model.to(device=device)
hf_reference_model.to(device=device)
hf_base_model.train()
hf_reference_model.eval()
hf_trainer.model = hf_base_model
hf_trainer.reference_model = hf_reference_model
print("policy device:    ", next(hf_base_model.parameters()).device)
print("reference device: ", next(hf_reference_model.parameters()).device)
print("trainer device:   ", hf_trainer.args.device)

# train
hf_trainer.train()                                                                              # option 1: train from scratch (vanilla checkpoint)
#hf_trainer.train(resume_from_checkpoint=True)                                                  # option 2: resume from latest available checkpoint
#hf_trainer.train(resume_from_checkpoint=os.path.join(DIR_WHISPER_OUTPUT, "checkpoint-855"))    # option 3: resume from specific checkpoint

# compute time
end_time = datetime.now()
time_delta = end_time - start_time

# compute training duration
metadata_training_duration_seconds  = time_delta.total_seconds()
metadata_training_duration_hours    = metadata_training_duration_seconds / 3600

# cleanup
try:
    del start_time, end_time, time_delta
    del metadata_training_duration_seconds
except Exception:
    pass


In [ ]:
# Empty GPU RAM
import torch

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("GPU RAM emptied.")
else:
    print("No GPU available, no GPU RAM to empty.")

# Move models to the selected device (GPU)
device = torch.device("cuda:0")

## Evaluate model post-training

In [ ]:
# train split
# CAREFUL:  these results will not be representative of model's performance on unseen / new data
#           (because both these datasets have been seen during training!)

'''
print('Evaluating best model on "train" set, language:', parameter_model_language)
results_train = hf_trainer.evaluate(hf_dataset["train"], language=parameter_model_language)
print(results_train)
'''

In [ ]:
# test split
# CAREFUL:  these results will not be representative of model's performance on unseen / new data
#           (because both these datasets have been seen during training!)

'''
print('Evaluating best model on "test" set, language:', parameter_model_language)
results_test = hf_trainer.evaluate(hf_dataset["test"], language=parameter_model_language)
print(results_test)
'''

In [ ]:
# eval dataset
# This dataset has NOT been seen during training
# the results are representative of model's performance on unseen / new data

print('Evaluating best model on "eval" set, post-training, language:', parameter_model_language)
results_eval_post = hf_trainer.evaluate(hf_dataset["eval"], language=parameter_model_language)
print(results_eval_post)

# Save (model, metadata, results)

## Save model

In [ ]:
# Save the best model locally
# The trainer automatically saves the best model, so we just need to copy it

import shutil

# best model directory
!rm -rf {DIR_WHISPER_MODEL_OUTPUT}
!mkdir  {DIR_WHISPER_MODEL_OUTPUT}

# Metadata directory
DIR_METADATA    = os.path.join(DIR_WHISPER_MODEL_OUTPUT, "metadata")
!rm -rf {DIR_METADATA}
!mkdir  {DIR_METADATA}

# save model and processor, so we can later load as pretrained
hf_trainer.model.save_pretrained(DIR_WHISPER_MODEL_OUTPUT, safe_serialization=False)    # model
hf_processor.save_pretrained    (DIR_WHISPER_MODEL_OUTPUT, safe_serialization=False)    # processor

print(f"✓ Best model saved to: {DIR_WHISPER_MODEL_OUTPUT}")

# List saved files
saved_files = os.listdir(DIR_WHISPER_MODEL_OUTPUT)
print(f"✓ Files saved: {saved_files}")

# cleanup
del saved_files

## Save results

In [ ]:
# Metadata directory
DIR_METADATA                = os.path.join(DIR_WHISPER_MODEL_OUTPUT, "metadata")
DIR_METADATA_MODEL          = os.path.join(DIR_METADATA, "model")
DIR_METADATA_DATASETS       = os.path.join(DIR_METADATA, "datasets")
DIR_METADATA_TENSORBOARD    = os.path.join(DIR_METADATA, "tensorboard")
!rm -rf {DIR_METADATA}
!mkdir  {DIR_METADATA}
!mkdir  {DIR_METADATA_MODEL}
!mkdir  {DIR_METADATA_DATASETS}
!mkdir  {DIR_METADATA_TENSORBOARD}

## Save model metadata

In [ ]:
# Save training arguments file

hf_trainer_args_dict = hf_trainer_args.to_dict()

# Define the path for the training arguments file
hf_trainer_args_file_path = os.path.join(DIR_METADATA_MODEL, 'trainer_parameters.json')

# Save the dictionary to a JSON file for human-readable format
with open(hf_trainer_args_file_path, 'w') as f:
    json.dump(hf_trainer_args_dict, f, indent=4)

print(f"Training arguments saved to: {hf_trainer_args_file_path}")

# cleanup
del f

In [ ]:
# Save evaluation results

import json

# Save evaluation results

results_test_file_path = os.path.join(DIR_METADATA_MODEL, 'results_trainer.json')

# Combine results into a single dictionary structure
evaluation_results = {
#    "train":    results_train,
#    "test":     results_test,
    "eval":     results_eval_post
}

# Write the evaluation results to the file in JSON format
with open(results_test_file_path, 'w', encoding='utf-8') as f:
    json.dump(evaluation_results, f, indent=4, ensure_ascii=False)

print(f"Evaluation results saved to: {results_test_file_path}")

# cleanup
del f, evaluation_results
#del results_test, results_train, results_test_file_path

In [ ]:
# simplify basemodel name
base_model_name = parameter_basemodel_path.rsplit('/', 1)[-1]
print(base_model_name)

In [ ]:
# Save Metadata file for model

# Get current time in HHhMMm format
metadata_training_date  = datetime.now().strftime('%Y%m%d')
metadata_finish_time    = datetime.now().strftime('%Hh%Mm')

# get info on GPU
if torch.cuda.is_available():
    metadata_gpu_name = torch.cuda.get_device_name(0)
    print(f"GPU Type: {metadata_gpu_name}")
else:
    metadata_gpu_name = "n/a (no GPU found, or CUDA not available)"

# Build field history: prepend previous values if the base checkpoint
# already has a metadata_model.json (keeps a chain of all ancestors)
metadata_base_model         = base_model_name
metadata_training_technique = "Kahneman-Tversky Optimization (KTO)"

prev_metadata_path = os.path.join(parameter_basemodel_path, "metadata", "metadata_model.json")
if os.path.isfile(prev_metadata_path):
    with open(prev_metadata_path, "r") as f:
        prev_metadata = json.load(f)
    
    prev_base_model = prev_metadata.get("base_model(s)")
    if prev_base_model:
        metadata_base_model = f"{prev_base_model} > {base_model_name}"
        print(f"Base model history: {metadata_base_model}")
    
    prev_training_technique = prev_metadata.get("training_technique(s)")
    if prev_training_technique:
        metadata_training_technique = f"{prev_training_technique} > {metadata_training_technique}"
        print(f"Training technique history: {metadata_training_technique}")

# Create a dictionary with desired metadata
metadata = {
    "base_model(s)":                        metadata_base_model,
    "training_technique(s)":                metadata_training_technique,
    "training_date":                        metadata_training_date,
    "training_finish_time":                 metadata_finish_time,
    "training_duration_hours":              f"{metadata_training_duration_hours:.2f}",
    "model_language":                       parameter_model_language,
    #"model_type":                           parameter_basemodel_path,
    "model_n_mel_bins":                     parameter_model_n_mel_bins,
    "model_task":                           parameter_model_task,
    "model_metric":                         hf_trainer_args.metric_for_best_model,
    "trainer_batch_size":                   parameter_trainer_batch_size,
    "trainer_learning_rate":                parameter_trainer_learning_rate,
    "trainer_warmup_steps":                 parameter_trainer_warmup_steps,
    "trainer_max_gen_len":                  parameter_trainer_max_gen_len,
    "trainer_n_epochs":                     parameter_trainer_n_epochs,
    "trainer_update_decoder":               parameter_trainer_update_decoder,
    "trainer_update_encoder":               parameter_trainer_update_encoder,
    "trainer_update_projection":            parameter_trainer_update_projection,
    "trainer_kto_beta":                     parameter_trainer_kto_beta,
    "trainer_desirable_weight":             parameter_trainer_desirable_weight,
    "trainer_undesirable_weight":           parameter_trainer_undesirable_weight,
    "trainer_kto_kl_ema_decay":             parameter_trainer_kto_kl_ema_decay,
    "trainer_kto_mask_special_tokens":      parameter_trainer_kto_mask_special_tokens,
    "gpu_name":                             metadata_gpu_name
}

# Define the path for the JSON file
metadata_file_path = os.path.join(DIR_METADATA_MODEL, 'metadata_model.json')

# Save the dictionary to a JSON file
with open(metadata_file_path, 'w') as f:
    json.dump(metadata, f, indent=4)

print(f"Model metadata saved to: {metadata_file_path}")

# cleanup
del f, metadata_file_path, metadata, metadata_base_model, metadata_training_technique, prev_metadata_path


In [ ]:
# get latest "trainer_state.json" from checkpoint folders
# and remove all temp checkpoints

# Find the latest checkpoint directory
latest_checkpoint_dir   = None
target_checkpoint_path  = None
max_checkpoint_num      = -1

# List contents of OUTPUT_DIR
output_dir_contents = os.listdir(DIR_WHISPER_OUTPUT)
print(DIR_WHISPER_OUTPUT)
print(output_dir_contents)

for item in output_dir_contents:
    if item.startswith('checkpoint-') and os.path.isdir(os.path.join(DIR_WHISPER_OUTPUT, item)):
        print(f'Found checkpoint folder: {item}')
        try:
            checkpoint_num = int(item.split('-')[1])
            if checkpoint_num > max_checkpoint_num:
                max_checkpoint_num = checkpoint_num
                latest_checkpoint_dir = item
        except ValueError:
            # Not a valid checkpoint number, ignore
            pass

if latest_checkpoint_dir:
    source_checkpoint_path = os.path.join(DIR_WHISPER_OUTPUT, latest_checkpoint_dir)
    print(f"Found latest checkpoint folder: {source_checkpoint_path}")

    # Define the target path, naming the folder 'checkpoint'
    target_checkpoint_path = os.path.join(DIR_WHISPER_OUTPUT, 'checkpoint_latest')

    # Ensure the target directory in Google Drive exists and is empty for a clean copy
    if os.path.exists(target_checkpoint_path):
        shutil.rmtree(target_checkpoint_path)
    os.makedirs(target_checkpoint_path, exist_ok=True)

    # Copy only .json files from source to target
    if os.path.exists(source_checkpoint_path):
        copied_files_count = 0
        json_files_copied = []
        for file_name in os.listdir(source_checkpoint_path):
            source_file_path = os.path.join(source_checkpoint_path, file_name)
            if os.path.isfile(source_file_path) and file_name.endswith('.json'):
                target_file_path = os.path.join(target_checkpoint_path, file_name)
                shutil.copy2(source_file_path, target_file_path)
                copied_files_count += 1
                json_files_copied.append(file_name)

        if copied_files_count > 0:
            print(f"Copied {copied_files_count} JSON files ({', '.join(json_files_copied)}) from checkpoint folder to: {target_checkpoint_path}")
        else:
            print(f"No JSON files found to copy from {source_checkpoint_path}.")
    else:
        print(f"Error: Checkpoint folder not found at {source_checkpoint_path}")
else:
    print(f"No checkpoint directory found in {DIR_WHISPER_OUTPUT}. Skipping checkpoint folder save.")

# remove all files except "trainer_state.json" from target_checkpoint_path
if os.path.exists(target_checkpoint_path):
    for file_name in os.listdir(target_checkpoint_path):
        if file_name != "trainer_state.json":
            os.remove(os.path.join(target_checkpoint_path, file_name))

# move "trainer_state.json"
source_file_path = os.path.join(target_checkpoint_path, "trainer_state.json")
target_file_path = os.path.join(DIR_METADATA_MODEL, "trainer_state.json")
shutil.move(source_file_path, target_file_path)
print(f'Moved "trainer_state.json" to: {target_file_path}')

## Save tensorboard files

In [ ]:
# tensorboard files

# copy content of DIR_WHISPER_OUTPUT/runs to DIR_METADATA/tensorboard
shutil.copytree(os.path.join(DIR_WHISPER_OUTPUT, "runs"), DIR_METADATA_TENSORBOARD, dirs_exist_ok=True)
print(f"Tensorboard logs copied to: {DIR_METADATA_TENSORBOARD}")

## Save dataset(s) metadata

In [ ]:
from huggingface_hub import hf_hub_download
import shutil

for dataset_id in HF_DATASETS_TRAINING:
    dataset_name = dataset_id.rsplit("/", 1)[-1]

    #
    print(f"Processing dataset: {dataset_id}")
    
    # metadata.json
    metadata_path = hf_hub_download(
        repo_id     = dataset_id,
        filename    = "data.json",
        repo_type   = "dataset",
    )
    shutil.copy2(metadata_path, os.path.join(DIR_METADATA_DATASETS, f"ds_{dataset_name}_metadata.json"))
    print(f"Copied metadata for {dataset_id}")

    # data.json
    data_path = hf_hub_download(
        repo_id     = dataset_id,
        filename    = "data.json",
        repo_type   = "dataset",
    )
    shutil.copy2(data_path, os.path.join(DIR_METADATA_DATASETS, f"ds_{dataset_name}_data.json"))
    print(f"Copied data for {dataset_id}")   

# Evaluate

This script does not automatically push to the HF hub.

This is to avoid automatically uploading models which performance have not been manually assessed / confirmed by a human developer.

The developer is supposed to assess the performance of the trained model against old models (e.g. through the script `models-evaluator/models-evaluator.ipynb`)

If performace are satisfactory, one can then manually upload the model through script `utils/push-model-to-hf-hub.py`

# Push to HF hub

This script does not automatically push to the HF hub.
This is to avoid automatically uploading models which performacne have not been manually assessed / confimred by a human developer.
The developer is supposed to assess the performance of the trained model against old models (e.g. through the script `models-evaluator/models-evaluator.ipynb`)

If performace are satisfactory, one can then manually uplaod the model through script `utils/push-model-to-hf-hub.py`

# Cleanup

In [ ]:
# todo (see SFT script)